# LLM-as-Judge: Biotech Lyric Battles

This notebook evaluates four frontier language models on creative songwriting tasks at the intersection of biotechnology and music.

The project has three layers:

1. Generate lyrics from four models across fourteen biotech/music prompts.
2. Use the same models as judges in anonymous and labelled modes.
3. Compare model judgements against human ratings to analyse bias, disagreement, and taste.

This notebook is the main project artefact. Code, outputs, charts, and commentary will live together here.

## 1. Setup and configuration

This section loads the Python packages, checks API keys, and imports helper functions from `src/`.

Run these cells once at the start of each notebook session.

In [1]:
import json
import os
import sys
from pathlib import Path

import anthropic
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm.notebook import tqdm
from xai_sdk import Client
from xai_sdk.chat import user

# Make sure the notebook can import from the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/danieljohnson/Desktop/modeleval


In [2]:
# Load API keys from .env
load_dotenv(PROJECT_ROOT / ".env")

required_keys = [
    "ANTHROPIC_API_KEY",
    "OPENAI_API_KEY",
    "XAI_API_KEY",
]

missing_keys = [key for key in required_keys if not os.getenv(key)]

if missing_keys:
    raise ValueError(f"Missing API keys: {missing_keys}")

print("All API keys loaded OK")

All API keys loaded OK


In [3]:
from src.models import call_opus, call_haiku, call_gpt5, call_grok

print("Model wrapper imports OK")

Model wrapper imports OK


### Smoke test

This cell sends a tiny prompt to all four models. It proves that the API keys, SDKs, and model wrappers work before we spend money on the real generation run.

In [4]:
test_prompt = "Write one line about ATP."

model_calls = {
    "Claude Opus 4.7": call_opus,
    "Claude Haiku 4.5": call_haiku,
    "GPT-5.5": call_gpt5,
    "Grok 4.3": call_grok,
}

for model_name, call_model in model_calls.items():
    print(f"\n--- {model_name} ---")
    response = call_model(test_prompt, max_tokens=500)
    print(response)


--- Claude Opus 4.7 ---
ATP (adenosine triphosphate) is the primary energy-carrying molecule used by cells to power biological processes.

--- Claude Haiku 4.5 ---
ATP (adenosine triphosphate) is the cell's primary energy currency, releasing energy when its high-energy phosphate bonds are broken to power cellular processes.

--- GPT-5.5 ---
ATP (adenosine triphosphate) is the primary energy-carrying molecule that powers many cellular processes.

--- Grok 4.3 ---
ATP (adenosine triphosphate) is the primary energy currency that powers cellular processes through hydrolysis of its high-energy phosphate bonds.


## 2. The eval set

This section loads the 14 prompt eval set.

The prompts are split into four categories:

1. Genre x biotech topic
2. Battle / diss tracks
3. Tribute songs
4. Storytelling tracks

The point is not just to test whether models can write lyrics. The point is to force tradeoffs between genre fidelity, scientific accuracy, lyrical craft, cleverness, and commitment.

In [4]:
prompts_path = PROJECT_ROOT / "prompts" / "prompts.json"

with prompts_path.open("r", encoding="utf-8") as f:
    prompts = json.load(f)

prompts_df = pd.DataFrame(prompts)

print(f"Loaded {len(prompts_df)} prompts")
prompts_df

Loaded 14 prompts


,id,category,category_short,prompt,notes
0,A1,Genre x Biotech Topic,A,Write a UK grime track about a PCR that keeps ...,"Tests genre commitment, lab frustration, techn..."
1,A2,Genre x Biotech Topic,A,Write a 2000s hip hop track about leaving big ...,"Tests commercial biotech storytelling, career ..."
2,A3,Genre x Biotech Topic,A,Write a 90s boom bap rap about complementary D...,"Tests metaphor, molecular accuracy, and whethe..."
3,A4,Genre x Biotech Topic,A,Write a UK drill track about lab politics and ...,"Likely stress test for refusal, sanitisation, ..."
4,A5,Genre x Biotech Topic,A,Write a trap song about a Western blot that wi...,"Tests technical references to blotting, frustr..."
5,B1,Battle / Diss Track,B,Write a rap battle between small molecule drug...,"Tests ability to represent both sides, use rea..."
6,B2,Battle / Diss Track,B,Write a rap battle between wet lab scientists ...,"Tests humour, domain fluency, stereotypes, and..."
7,B3,Battle / Diss Track,B,Write a rap battle between a principal investi...,"Tests academic lab culture knowledge, characte..."
8,B4,Battle / Diss Track,B,Write a rap battle between short-read sequenci...,"Tests technical accuracy around NGS tradeoffs,..."
9,C1,Tribute Song,C,Write a genuine love song to ATP.,Tests whether the model can find emotional wei...


## 3. Generation prompt preview

Before calling the models, we preview the exact instruction that will be sent to them.

This matters because weak prompts produce generic outputs, and then the evaluation measures prompt laziness rather than model behaviour.

In [5]:
from src.generate import build_generation_prompt

example_prompt = build_generation_prompt(prompts[0])

print(example_prompt)

You are writing original song lyrics for a creative AI evaluation.

TASK:
Write a UK grime track about a PCR that keeps failing.

CONTEXT:
This is part of an eval comparing language models on creative writing at the intersection of biotechnology and music.

WHAT TO PRODUCE:
Write complete song lyrics for the task above.

REQUIREMENTS:
- Commit strongly to the requested genre or format.
- Use accurate biotechnology or life science references.
- Make the lyrics specific, not generic.
- Use rhyme, rhythm, structure, and memorable lines.
- Avoid bland motivational science lyrics.
- Avoid explaining the song. Only output the lyrics.
- Do not include commentary before or after the lyrics.

HELPFUL NOTES:
Tests genre commitment, lab frustration, technical references to PCR failure modes, and ability to avoid generic science lyrics.

LENGTH:
Aim for roughly 24 to 40 lines.


## 4. Single-prompt generation test

Before running the full 56-output generation step, this cell tests the generation pipeline on one prompt across all four models.

This should produce 4 outputs total.

In [5]:
from src.generate import run_generation

test_generations_path = PROJECT_ROOT / "data" / "test_generations.json"

test_prompts = prompts[:1]

test_model_calls = {
    "Claude Opus 4.7": call_opus,
    "Claude Haiku 4.5": call_haiku,
    "GPT-5.5": call_gpt5,
    "Grok 4.3": call_grok,
}

test_generations = run_generation(
    prompts=test_prompts,
    model_calls=test_model_calls,
    output_path=test_generations_path,
    max_tokens=1200,
    overwrite=False,
)

test_generations_df = pd.DataFrame(test_generations)

print(f"Loaded {len(test_generations_df)} test outputs")
test_generations_df[["prompt_id", "model", "output_text"]]

Generation file already exists. Loading existing results from: /Users/danieljohnson/Desktop/modeleval/data/test_generations.json
Loaded 4 test outputs


,prompt_id,model,output_text
0,A1,Claude Opus 4.7,"**""NO BANDS"" — 140 BPM, gully**\n\nYo —\nPipet..."
1,A1,Claude Haiku 4.5,"# PCR TING\n\n[INTRO]\nYo, yo, yo\nLab still c..."
2,A1,GPT-5.5,"[Intro] \nYo, lab coat zipped, gloves on tigh..."
3,A1,Grok 4.3,[Hook]\nPCR cycle spinnin' but the band don't ...


### Inspect test generations

The JSON file is for storage, not reading. This cell prints the generated lyrics in a readable format so we can quickly judge whether the generation prompt is working.

In [ ]:
from IPython.display import Markdown, display

for row in test_generations:
    display(Markdown(f"""
---

### {row["model"]}

**Prompt:** {row["prompt"]}

```text
{row["output_text"]}
```
"""))

Prompt ID: A1
Model: Claude Opus 4.7
----------------------------------------------------------------------------------------------------
**"NO BANDS" — 140 BPM, gully**

Yo —
Pipette in my left, Eppendorf right,
Master mix made, been here all night,
Ninety-eight degrees, lid sealed tight,
Bruv, why's my gel lookin' empty? No light.

Primer dimer crew tryna gatecrash the rave,
Forty cycles deep and I'm halfway in the grave,
Loaded up the ladder, where my amplicon stays?
Just a smear and a smudge and a "back to square A."

Mg2+, did I add ya? Did I not?
Template DNA, did I plate it? Forgot.
Annealing at sixty — too high, too hot —
Touchdown PCR, still I'm getting nought.

GC-rich region, secondary structure,
DMSO splash, betaine, brudda,
Q5 polymerase, Phusion, ya muvver,
Still no band, swear down I'ma suffer.

Contamination — the negative's glowing,
Aerosol drift and the bench is overflowing,
Filter tips changed but the panic keeps growing,
PI's at the door and the deadline's showing.
